## Container & build troubleshooting playbook

Most container failures fall into a few shapes, each with a first set of commands. The reflex is always the same: **status → logs → inside.**

**"It won't start" / "starts then exits":**

```bash
docker ps -a --filter name=myc      # what's its STATUS + exit code?
docker logs myc                     # what did the main process say?
docker inspect myc | jq '.State'    # exit code, OOMKilled flag, timestamps
```

Exit **137** = OOM-killed (raise `--memory`); **125/126/127** = the run command, or the container's command, was wrong (module 03's exit codes). If logs don't reveal it, **boot a shell instead of the app** to poke around:

```bash
docker run --rm -it --entrypoint sh IMAGE     # bypass the entrypoint
```

**"Running but the app is unhealthy":** get inside and look.

```bash
docker exec -it myc sh              # a shell in the live container
docker exec myc ps -ef              # what's actually running?
docker exec myc netstat -tlnp       # is it listening where you think?
docker stats --no-stream myc        # CPU / memory / I/O right now
```

**Build failures** debug through BuildKit's plumbing — the key flag is **`--progress=plain`**, which un-collapses the log so you can see each step's cache state:

```bash
docker buildx build --progress=plain -t myimg .
docker builder prune                # clear a poisoned build cache
```

The usual culprits map back to module 02: **cache misses where you expected hits** (a `COPY` source changed, or a stale `.dockerignore` is leaking files into the context); a **slow `COPY .`** (huge context — `du -sh .`, then trim with `.dockerignore`); and `failed to compute cache key: … not found` (a source path that isn't in the context at all). Almost every build problem is really a *context* or *cache* problem — which is why `.dockerignore` and instruction ordering, from module 02, prevent most of them before they start.